# Groundhog — notebook demo
A lightweight, **real** experiment loop: we train a small softmax classifier across a few configs and watch Groundhog memory fill up — Pre-flight Guard, semantic recall, and derived insights — all scoped to one project.

Prereqs: the 3 services running (cognee :8010, backend :8000, frontend :5173).

In [1]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=='pages' else os.getcwd())
# make sure repo root is importable so `demo` package + sdk resolve
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)
from demo._bootstrap import init, BACKEND
from demo.train_lib import train_and_evaluate, DATASET_INFO
import groundhog

project_id = init(experiment='blobs3_notebook')
print('connected to project:', project_id)

connected to project: proj_blobs3_sgd_demo_0d78c7c9


### Register the Groundhog notebook magics
`%groundhog_hypothesis` and `%groundhog_decision` write the *why* straight into project memory.

In [2]:
get_ipython().run_line_magic('load_ext', 'connectors.jupyter_magic')


[+] Groundhog Researcher Insight Engine Registered.
    Available macros:
      - %groundhog_hypothesis <statement>
      - %groundhog_decision <rationale>
      - %%groundhog_watch  (Place at top of code cells)
    Note: Initialize the SDK (`groundhog.init(project_id=...)`) first
    so these magics instantly sync to your live Cognee memory graph.



### Dataset (proper, versioned input)

In [3]:
DATASET_INFO

{'name': 'blobs3',
 'version': 'v1',
 'preprocessing': 'z-score standardized (per-feature mean/std from train split)',
 'split_rationale': '70/30 stratified train/val, fixed seed 7',
 'quality_issues': 'class 2 slightly overlaps class 1 (intentional, ~4% Bayes error)'}

### Hypothesis → memory

In [4]:
%groundhog_hypothesis A little weight decay should improve val_accuracy on blobs3 without hurting convergence.

[+] Groundhog instantly logged hypothesis to project 'proj_blobs3_sgd_demo_0d78c7c9': 'A little weight decay should improve val_accuracy on blobs3 without hurting convergence.'


### Run a small weight-decay sweep
`groundhog.run(...)` runs the Pre-flight Guard on enter and auto-logs metrics + wall-clock on exit. Re-running this cell should show the guard flag configs you already tried.

In [5]:
EXPERIMENT = 'blobs3_notebook'
for wd in (0.0, 1e-3, 1e-2):
    cfg = {'model':'softmax','optimizer':'sgd','lr':0.1,'batch_size':32,'weight_decay':wd,'epochs':20}
    with groundhog.run(config=cfg, experiment=EXPERIMENT) as r:
        if r.already_tried:
            print(f'wd={wd}: already tried -> {r.prior_metrics}; skipping'); r.skip()
        else:
            out = os.path.join('outputs', EXPERIMENT, f'wd_{wd}')
            m = train_and_evaluate(cfg, out_dir=out)
            r.log(**m)
            print(f'wd={wd}: val_acc={m["val_accuracy"]} val_loss={m["val_loss"]}')

[Groundhog] 🛡️ Checking Pre-flight Guard for config...


[Groundhog] ⚠️ This config was already tried (match=exact). Prior result: {'val_accuracy': 0.6364, 'val_loss': 0.8088, 'train_loss': 0.81697, 'epochs': 20}
[Groundhog] Check `run.already_tried` to skip, or call `run.skip()`.
wd=0.0: already tried -> {'val_accuracy': 0.6364, 'val_loss': 0.8088, 'train_loss': 0.81697, 'epochs': 20}; skipping
[Groundhog] 🛡️ Checking Pre-flight Guard for config...


wd=0.001: val_acc=0.6364 val_loss=0.8089


[Groundhog] 🛡️ Checking Pre-flight Guard for config...


wd=0.01: val_acc=0.6364 val_loss=0.81


### Record a decision

In [6]:
%groundhog_decision Keeping weight_decay small (<=1e-3); 1e-2 underfits blobs3.

[+] Groundhog instantly logged decision to project 'proj_blobs3_sgd_demo_0d78c7c9': 'Keeping weight_decay small (<=1e-3); 1e-2 underfits blobs3.'


### Ask the memory

In [7]:
print(groundhog.query('Across the notebook runs, which weight_decay gave the best val_accuracy and why?'))

The weight decay that gave the best validation accuracy in the experiments was **0.0**, associated with a learning rate of **0.5**. This configuration achieved a validation accuracy of **0.6566** during the `blobs3_lr_sweep` experiment. 

The choice of a weight decay value of 0.0 was justified because it allowed for better performance by minimizing potential overfitting effects that could arise from introducing regularization, which had shown low impact in prior runs. Additionally, the learning rate of 0.5 was established as the optimal value, contributing significantly to the model's convergence and validation accuracy.


### What the memory learned (derived insights)

In [8]:
import json, urllib.request
url = f'{BACKEND}/api/insights?project_id={project_id}'
ins = json.load(urllib.request.urlopen(url, timeout=60))
print('summary:', ins.get('summary'))
for s in ins.get('parameter_sensitivity', [])[:5]:
    print(f"  {s['parameter']}: impact={s['sensitivity']} best={s['best_value']}")

summary: Across 6 completed runs: The most impactful hyperparameter is 'lr' (moves val_accuracy by ~0.0354); best value seen: 0.5. 'weight_decay' had the least effect (0.0013). Best on blobs3: val_accuracy=0.6566.
  lr: impact=0.0354 best=0.5
  weight_decay: impact=0.0013 best=0.01


Open the dashboard at **http://localhost:5173** and select this project — the runs, artifacts, and insights you just created are all there.